# RVC WebUI 一键笔记本（Python 3.12 + NVIDIA CUDA 12.8）

这个笔记本会在**笔记本内部**完成下面所有步骤，并在笔记本里直接启动 RVC 的 WebUI（推理 / 人声分离 / 多说话人训练 / MSST 伴奏分离 / 常见问题等全部页面），同时生成可公网访问的地址。

1. 克隆仓库 `miniworldRuman/Retrieval-based-Voice-Conversion-WebUI`
2. 下载模型 / 完整安装包 `RVC20260723Nvidia.7z`（≈7.8 GB，来自 ModelScope `FlowerCry/rvc-windows-packages`）
3. 解压到 `/tmp`
4. 把安装包内的 `assets` 与 `logs` 复制合并进仓库
5. 两阶段 pip 安装依赖（torch cu128 → `requirments_cu128_py312.txt`）
6. 在笔记本内启动 WebUI，并给出本地/公网访问地址

> 前置要求：
> - Python 3.12 x64（`requirments_cu128_py312.txt` 面向 3.12）
> - NVIDIA 驱动支持 CUDA 12.8（镜像可换官方源）
> - 磁盘剩余 ≥ 30 GB（下载 7.8 GB + 解压 + torch）
>
> 所有下载都支持断点，重新执行对应单元即可续传。


## 单元执行顺序

| 单元 | 内容 |
| --- | --- |
| 帮助函数 | `run` / `pip_install` / `show_tail` 等小工具 |
| 参数配置 | 仓库、安装包、端口、隧道方式等（按需修改） |
| 环境检查 | Python / GPU / 磁盘空间 |
| 系统工具与 Python 库 | ffmpeg、aria2、7z、py7zr 等 |
| 克隆仓库 | 克隆 / 更新 `miniworldRuman` 仓库 |
| 下载完整包 | 下载 7.8 GB 的 `.7z` 到 `/tmp` |
| 解压到 /tmp | 7z / py7zr 解压到 `/tmp/rvc_extract` |
| 复制合并 assets 与 logs | 自动定位安装包根目录并合并 |
| 校验模型与回退下载 | 校验核心模型，缺失时可补齐 |
| 安装依赖 | torch cu128 + requirements |
| 环境自检 | 导入 torch / gradio |
| 启动 WebUI | 后台启动 + 本地/公网地址 |
| 查看状态 / 停止 | `webui_status()` / `show_webui_log()` / `stop_webui()` |


In [ ]:
# 帮助函数
import os, sys, shutil, socket, re, time, glob, json, subprocess, pathlib, platform, hashlib, urllib.request
from pathlib import Path


def run(cmd, check=True, timeout=None):
    """执行 shell 命令并打印输出（不依赖 ! 魔法）。"""
    print(">>> " + cmd, flush=True)
    r = subprocess.run(
        cmd, shell=True, executable="/bin/bash", text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout,
    )
    if r.stdout and r.stdout.strip():
        print(r.stdout.strip()[-6000:])
    if check and r.returncode != 0:
        raise RuntimeError("命令失败 (rc=%s): %s" % (r.returncode, cmd))
    return r


def pip_install(req, extra=""):
    """pip 安装，首次失败自动加 --break-system-packages 重试。"""
    cmd = '"%s" -m pip install --disable-pip-version-check -q %s %s' % (sys.executable, extra, req)
    r = run(cmd, check=False)
    if r.returncode != 0:
        print("[pip] 重试：添加 --break-system-packages")
        run(cmd + " --break-system-packages", check=False)


def show_tail(path, n=80):
    p = Path(path)
    if not p.exists():
        print("文件不存在：", path)
        return
    lines = p.read_text(errors="replace").splitlines()
    print("\n".join(lines[-n:]))


def is_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def is_port_open(port, host="127.0.0.1"):
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False


def merge_tree(src, dst):
    """把 src 目录合并进 dst（跳过大小相同的同名文件，视为已有）。"""
    src, dst = Path(src), Path(dst)
    copied = 0
    total_bytes = 0
    for s in src.rglob("*"):
        if not s.is_file():
            continue
        rel = s.relative_to(src)
        d = dst / rel
        try:
            if d.exists() and d.stat().st_size == s.stat().st_size:
                continue
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(s, d)
            copied += 1
            total_bytes += s.stat().st_size
        except OSError as e:
            print("  跳过", rel, e)
    return copied, total_bytes


print("帮助函数已加载")

In [ ]:
# 参数配置（按需修改）
REPO_URL = "https://github.com/miniworldRuman/Retrieval-based-Voice-Conversion-WebUI.git"
WORK_DIR = os.environ.get("RVC_WORK_DIR", "/root/RVC")   # 项目目录（Colab 可改为 /content/RVC）

PKG_NAME   = "RVC20260723Nvidia.7z"
PKG_URL    = "https://www.modelscope.cn/models/FlowerCry/rvc-windows-packages/resolve/master/" + PKG_NAME
PKG_PATH   = "/tmp/" + PKG_NAME                           # 完整包放到 /tmp
PKG_SHA256 = "278cd7db9c9e86e45b506d70959297b0c56802d6d79d6d6481b2b6b0f925be86"
VERIFY_SHA256  = False                                    # 大文件 sha256 校验很耗时，确需校验改为 True
EXTRACT_DIR = "/tmp/rvc_extract"                          # 解压目录（/tmp 下）

PORT          = 7865                                       # WebUI 端口
PUBLIC_TUNNEL = os.environ.get("RVC_TUNNEL", "cloudflared")  # cloudflared / ngrok / none
NGROK_TOKEN   = os.environ.get("NGROK_TOKEN", "")
CLOUDFLARED_URL = os.environ.get("CLOUDFLARED_URL", "")     # 自定义 cloudflared 二进制下载地址

TORCH_INDEX = "https://mirrors.nju.edu.cn/pytorch/whl/cu128"  # 国内镜像；也可换官方 https://download.pytorch.org/whl/cu128
PYPI_INDEX  = "https://mirrors.pku.edu.cn/pypi/simple"
FALLBACK_DOWNLOAD = True                                   # 模型缺失时从网上下载补齐

print("配置完成  工作目录：", WORK_DIR)
print("安装包    ：", PKG_NAME, "->", PKG_PATH)

In [ ]:
# 环境检查
print("Python 版本 ：", sys.version.split()[0], " (目标 Python 3.12)")
print("平台        ：", platform.platform(), "|", platform.machine())
if shutil.which("nvidia-smi"):
    run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False)
else:
    print("[提示] 未找到 nvidia-smi，推理/训练将回退到 CPU。")
for d in ("/", "/tmp"):
    u = shutil.disk_usage(d)
    print("磁盘 %-6s 剩余 %8.1f GB" % (d, u.free / 1e9))
if Path(WORK_DIR).exists():
    print("[提示] %s 已存在，" % WORK_DIR, "克隆单元会 git pull 更新。")

In [ ]:
# 安装系统工具与 Python 库
run("apt-get update -qq >/dev/null 2>&1 || true", check=False)
run("apt-get install -y -qq git ffmpeg aria2 p7zip-full libportaudio2 libsndfile1 >/dev/null 2>&1 || true", check=False)
pip_install("py7zr huggingface_hub tqdm requests pyngrok")

print("git      :", shutil.which("git"))
print("ffmpeg   :", shutil.which("ffmpeg"))
print("aria2c   :", shutil.which("aria2c"))
print("7z       :", shutil.which("7z") or shutil.which("7zz") or "(将使用 py7zr)")

In [ ]:
# 克隆仓库
if not Path(WORK_DIR).exists():
    Path(WORK_DIR).parent.mkdir(parents=True, exist_ok=True)
    run("git clone --depth 1 %s %s" % (REPO_URL, WORK_DIR))
else:
    run("git -C %s pull --ff-only" % WORK_DIR, check=False)

if shutil.which("git-lfs"):
    run("git -C %s lfs pull 2>/dev/null || true" % WORK_DIR, check=False)

os.chdir(WORK_DIR)
print("工作目录：", os.getcwd())
print("仓库内容：", ", ".join(sorted(os.listdir("."))))

In [ ]:
# 下载完整安装包 (~7.8 GB) 到 /tmp，支持断点续传
def download_big(url, dest, expected_sha256=""):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists():
        if not expected_sha256:
            print("文件已存在（跳过）：", dest, "(%.2f GB)" % (dest.stat().st_size / 1e9))
            return
        h = hashlib.sha256()
        with open(dest, "rb") as f:
            for b in iter(lambda: f.read(8 * 1024 * 1024), b""):
                h.update(b)
        if h.hexdigest() == expected_sha256:
            print("文件已存在且 sha256 校验通过（跳过）：", dest)
            return
        print("sha256 不匹配，重新下载 ...")
        dest.unlink()

    aria2 = shutil.which("aria2c")
    if aria2:
        r = run(
            '"%s" -x 16 -s 16 -c --auto-file-renaming=false --allow-overwrite=true '
            '-d "%s" -o "%s" "%s"' % (aria2, dest.parent, dest.name, url),
            check=False,
        )
        if r.returncode == 0 and dest.exists():
            print("aria2 下载完成：%.2f GB" % (dest.stat().st_size / 1e9))
            return

    # urllib 断点续传回退
    part = dest.with_suffix(dest.suffix + ".part")
    headers = {"User-Agent": "Mozilla/5.0"}
    if part.exists():
        headers["Range"] = "bytes=%d-" % part.stat().st_size
    req = urllib.request.Request(url, headers=headers)
    resp = urllib.request.urlopen(req, timeout=60)
    if part.exists() and resp.status != 206:
        part.unlink()
    with resp, open(part, "ab" if part.exists() else "wb") as f:
        from tqdm import tqdm
        done = part.stat().st_size
        total = done + int(resp.headers.get("Content-Length") or 0)
        bar = tqdm(total=total, unit="B", unit_scale=True, desc=dest.name, initial=done)
        while True:
            chunk = resp.read(4 * 1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            bar.update(len(chunk))
        bar.close()
    os.replace(part, dest)
    print("urllib 下载完成：%.2f GB" % (dest.stat().st_size / 1e9))


_t0 = time.time()
print("开始下载完整安装包（约 7.8 GB）...")
download_big(PKG_URL, PKG_PATH, PKG_SHA256 if VERIFY_SHA256 else "")
print("下载耗时 %.1f 分钟； 文件：%s (%.2f GB)" % ((time.time() - _t0) / 60, PKG_PATH, Path(PKG_PATH).stat().st_size / 1e9))

In [ ]:
# 解压完整包到 /tmp/rvc_extract
work = Path(EXTRACT_DIR)
work.mkdir(parents=True, exist_ok=True)

seven = shutil.which("7z") or shutil.which("7zz")
ok = False
if seven:
    r = run('"%s" x -y -o"%s" "%s" >/dev/null 2>&1' % (seven, work, PKG_PATH), check=False)
    ok = r.returncode == 0
if not ok:
    print("使用 py7zr 解压（可能较慢，取决于 CPU）...")
    import py7zr
    with py7zr.SevenZipFile(str(PKG_PATH)) as z:
        z.extractall(path=str(work))

print("解压完成：", EXTRACT_DIR)

In [ ]:
# 定位安装包根目录，复制合并 assets 与 logs
cands = []
for p in Path(EXTRACT_DIR).rglob("*"):
    if (p.is_dir() and (p / "assets").is_dir() and (p / "logs").is_dir()
            and len(p.relative_to(EXTRACT_DIR).parts) <= 4):
        cands.append(p)

pkg_root = cands[0] if cands else Path(EXTRACT_DIR)
print("安装包根目录：", pkg_root)

for name in ("assets", "logs"):
    src = pkg_root / name
    dst = Path(WORK_DIR) / name
    if src.is_dir():
        c, b = merge_tree(src, dst)
        print("已合并 %-6s -> %-46s (%d 个文件, 共 %.2f GB)" % (name, dst, c, b / 1e9))
    else:
        print("[警告] 安装包中未找到 %s 目录：%s" % (name, src))

print("合并完成。")

In [ ]:
# 校验模型与回退下载
repo = Path(WORK_DIR)

critical = {
    "hubert": [
        repo / "assets/hubert_base/pytorch_model.bin",
        repo / "assets/hubert_base/config.json",
        repo / "assets/hubert_base/preprocessor_config.json",
    ],
    "rmvpe": [repo / "assets/rmvpe/rmvpe.pt"],
    "pretrain": [
        repo / "assets/pretrained/f0G40k.pth",
        repo / "assets/pretrained_v2/f0G40k.pth",
        repo / "assets/pretrained_v2/f0D40k.pth",
    ],
}
for name, files in critical.items():
    missing = [f for f in files if not f.is_file()]
    print("%-10s %s" % (name, "OK" if not missing else "缺失: " + ", ".join(str(f) for f in missing)))

weights = sorted((repo / "assets/weights").glob("*.pth")) if (repo / "assets/weights").is_dir() else []
indices = sorted((repo / "assets/indices").glob("*.index")) if (repo / "assets/indices").is_dir() else []
print("示例权重 (assets/weights) : %d 个 .pth" % len(weights))
print("示例索引 (assets/indices) : %d 个 .index" % len(indices))

pymss_root = repo / "assets/pymss_weights"
PYMSS_NEEDED = [
    "dereverb_mel_band_roformer_less_aggressive_anvuew_sdr_18.8050.ckpt",
    "dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt",
    "model_bs_roformer_ep_368_sdr_12.9628.ckpt",
    "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    "model_mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
]
pymss_missing = [n for n in PYMSS_NEEDED if not (pymss_root / n).is_file()]
print("PyMSS(MSST) 分离权重     ：%d/5，缺失：%s" % (len(PYMSS_NEEDED) - len(pymss_missing), ", ".join(pymss_missing) if pymss_missing else "无"))
mute_dir = repo / "logs" / "mute"
print("logs/mute（训练静音样本）: %d 个文件" % (sum(1 for _ in mute_dir.iterdir()) if mute_dir.is_dir() else 0))


if FALLBACK_DOWNLOAD:
    # ---- 1) 核心模型：lj1995/VoiceConversionWebUI（HuggingFace，可自动走 hf-mirror）----
    missing_core = [f for files in critical.values() for f in files if not f.is_file()]
    if missing_core:
        os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
        from huggingface_hub import hf_hub_download
        HF = "lj1995/VoiceConversionWebUI"
        jobs = [
            ("hubert_base/config.json",               repo / "assets/hubert_base"),
            ("hubert_base/preprocessor_config.json",  repo / "assets/hubert_base"),
            ("hubert_base/pytorch_model.bin",         repo / "assets/hubert_base"),
            ("rmvpe.pt",                              repo / "assets/rmvpe"),
            ("pretrained/f0G40k.pth",                 repo / "assets"),
            ("pretrained/f0D40k.pth",                 repo / "assets"),
            ("pretrained_v2/f0G40k.pth",              repo / "assets"),
            ("pretrained_v2/f0D40k.pth",              repo / "assets"),
        ]
        for fn, dst in jobs:
            target = dst / Path(fn).name
            if target.is_file():
                continue
            print("下载", fn, "...")
            hf_hub_download(HF, fn, local_dir=str(dst))

        # ---- mute 静音样本 ----
        if not mute_dir.is_dir() or not list(mute_dir.iterdir()):
            md = repo / ".model-downloads"
            md.mkdir(parents=True, exist_ok=True)
            print("下载 mute.zip ...")
            z = hf_hub_download(HF, "mute.zip", local_dir=str(md))
            run('"%s" -m zipfile -e "%s" "%s"' % (sys.executable, z, repo / "logs"), check=False)

    # ---- 2) pymss(MSST) 权重：baicai1145/pymss（ModelScope）----
    if pymss_missing:
        pymss_files = PYMSS_NEEDED + [
            "config_mel_band_roformer_karaoke.yaml",
            "dereverb_mel_band_roformer_anvuew.yaml",
            "model_bs_roformer_ep_317_sdr_12.9755.yaml",
            "model_bs_roformer_ep_368_sdr_12.9628.yaml",
        ]
        pymss_url_base = "https://www.modelscope.cn/models/baicai1145/pymss/resolve/master"
        for fn in pymss_files:
            target = pymss_root / fn
            if target.is_file():
                continue
            print("下载 pymss 权重", fn, "...")
            download_big(pymss_url_base + "/" + fn, target)

print("\n[完成] 模型校验与补齐结束。")

In [ ]:
# 安装依赖（两阶段）：阶段1 torch cu128，阶段2 requirements
pv = sys.version_info[:2]
print("当前 Python：%d.%d" % pv)
DO_STAGE1 = pv == (3, 12)
if not DO_STAGE1:
    print("[警告] requirments_cu128_py312.txt 面向 Python 3.12；当前为 %d.%d。\n"
          "       建议：conda create -n rvc python=3.12 -y && conda activate rvc\n"
          "       然后重新安装 jupyter 并重跑本笔记本。仍将继续尝试安装其余依赖。" % pv)

req = Path("requirments_cu128_py312.txt")
if not req.is_file():
    raise FileNotFoundError("未找到 %s，请先运行「克隆仓库」单元。" % req.name)

if DO_STAGE1:
    print("阶段 1/2：安装 torch / torchaudio（CUDA 12.8）...")
    pip_install(
        "torch==2.7.1+cu128 torchaudio==2.7.1+cu128",
        "--index-url %s --extra-index-url %s" % (TORCH_INDEX, PYPI_INDEX),
    )

print("阶段 2/2：安装项目依赖 %s ..." % req.name)
pip_install("-r " + req.name)
print("[完成] 依赖安装完成")

In [ ]:
# 环境自检
os.chdir(WORK_DIR)
import torch
print("torch     :", torch.__version__)
print("CUDA 可用 :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))
import gradio
print("gradio    :", gradio.__version__)
print("[完成] 核心库导入正常（部分告警可忽略）")

In [ ]:
# 启动 WebUI（后台）并生成公网访问地址
WEBUI_PORT = PORT
PHOME = "/tmp/rvc_webui"
Path(PHOME).mkdir(parents=True, exist_ok=True)
LOG  = Path(PHOME) / "webui.log"
CFL  = Path(PHOME) / "cloudflared.log"
PIDF = Path(PHOME) / "webui.pid"
URLF = Path(PHOME) / "public_url.txt"

# ---- 1) 启动 webui（如端口未占用）----
if is_port_open(WEBUI_PORT):
    print("WebUI 已在运行（端口 %d）。" % WEBUI_PORT)
else:
    if LOG.exists():
        LOG.unlink()
    p = subprocess.Popen(
        [sys.executable, "-u", "webui.py", "--port", str(WEBUI_PORT), "--noautoopen"],
        cwd=WORK_DIR,
        stdout=LOG.open("w"), stderr=subprocess.STDOUT,
        start_new_session=True, env=dict(os.environ),
    )
    PIDF.write_text(str(p.pid))
    print("WebUI 进程已启动，PID=%d" % p.pid)

# ---- 2) 等待端口就绪 ----
started = False
for _ in range(240):
    if is_port_open(WEBUI_PORT):
        started = True
        break
    if PIDF.exists() and not is_alive(int(PIDF.read_text().strip())):
        print("[错误] WebUI 进程已退出，日志尾部：")
        show_tail(LOG)
        break
    time.sleep(2)
print("本机访问地址： http://localhost:%d/" % WEBUI_PORT)
if not started:
    print("（如仍在启动可稍后重跑本单元；日志：%s）" % LOG)


# ---- 3) 公网隧道 ----
def _machine():
    m = platform.machine().lower()
    return "arm64" if m in ("aarch64", "arm64") else "amd64"


def start_cloudflared():
    cf = Path("/tmp/cloudflared")
    if not cf.exists():
        url = CLOUDFLARED_URL or "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-" + _machine()
        print("下载 cloudflared ...", url)
        urllib.request.urlretrieve(url, str(cf))
        os.chmod(cf, 0o755)
    if CFL.exists():
        CFL.unlink()
    subprocess.Popen(
        [str(cf), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:%d" % WEBUI_PORT],
        stdout=CFL.open("w"), stderr=subprocess.STDOUT, start_new_session=True,
    )
    url = None
    for _ in range(90):
        if CFL.exists():
            m = re.search(r"https://[A-Za-z0-9_.-]*?trycloudflare\.com", CFL.read_text(errors="replace"))
            if m:
                url = m.group(0)
                break
        time.sleep(2)
    if url:
        URLF.write_text(url)
    return url


def start_ngrok():
    if not NGROK_TOKEN:
        return None
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_TOKEN)
        u = ngrok.connect(WEBUI_PORT, bind_tls=True).public_url
        URLF.write_text(u)
        return u
    except Exception as e:
        print("[ngrok]", e)
        return None


public = URLF.read_text().strip() if URLF.exists() else ""
mode = str(PUBLIC_TUNNEL).lower()
if not public and mode in ("cloudflared", "ngrok"):
    if mode == "cloudflared":
        public = start_cloudflared()
        if not public:
            public = start_ngrok()
    else:
        public = start_ngrok()

print("\n================ WebUI 已就绪 ================")
print("  本地地址 : http://localhost:%d/" % WEBUI_PORT)
print("  公网地址 : " + (public if public else "（未生成，可改用 ngrok 或平台端口转发）"))
print("==============================================")

In [ ]:
# 查看状态 / 停止（可反复运行本单元）
def webui_status():
    print("端口 %d 状态：%s" % (PORT, "运行中" if is_port_open(PORT) else "未运行"))
    if PIDF.exists():
        pid = int(PIDF.read_text().strip())
        print("PID     ：", pid, "存活" if is_alive(pid) else "已退出")
        print("公网地址：", URLF.read_text().strip() if URLF.exists() else "无")


def show_webui_log(n=120):
    show_tail(LOG, n)


def stop_webui():
    if PIDF.exists():
        pid = int(PIDF.read_text().strip())
        try:
            os.killpg(pid, 9)
            print("已停止 WebUI（PID=%d）" % pid)
        except (ProcessLookupError, PermissionError):
            print("进程已不存在（PID=%d）" % pid)
    else:
        print("没有 PID 记录。")


webui_status()

## 使用提示

- 安装包自带的示例权重 `assets/weights/*.pth` 与索引 `logs/*.index` 已合并，可直接在「推理」页选择 `kikiV1` 等示例模型体验。
- 中国网络环境下 `trycloudflare.com` 可能访问不畅：可把配置改为 `PUBLIC_TUNNEL="ngrok"` 并填入 `NGROK_TOKEN`，或使用云平台的端口转发访问 `http://localhost:7865/`。
- 显存不足时，推理页关闭「半精度」；训练提示 f0method 需在有可用 CUDA 时用 `rmvpe`。
- 需要重启 WebUI：运行「查看状态 / 停止」单元里的 `stop_webui()`，再重新运行「启动 WebUI」单元。
- WebUI 日志在 `/tmp/rvc_webui/webui.log`，`show_webui_log()` 可查看。
